<a href="https://www.kaggle.com/code/jfaisal/track-a-b?scriptVersionId=340663038" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Soil Moisture Forecasting
**Track A: Statistical Baselines + Sequence Deep Learning**
**Track B: Graph-Based Modeling (GAT + T-GCN)**

Dataset: `agrovedha_ML_ready_v4_robust.csv` — 41,691 rows, 1-minute resolution, single continuous Bhopal location stream (no multi-site/sensor ID).

Key design decisions locked in during analysis (see markdown notes throughout):
- Task framed as **regression only** (forecast `soil_moisture_next_1m`); `irrigation_status` kept as an input feature despite near-zero variance (10/41,691 positive rows) — too rare to serve as a classification target.
- Provided `_scaled` predictor columns were found inconsistent with raw data and are **not used** — all scaling is refit per split.
- Track B graph nodes are the **6 measured variables** (Soil_Moisture, Temperature, Humidity, irrigation_status, Hour_sin, Hour_cos), not spatial sensors — this dataset has no site/sensor ID to support a spatial graph. Adjacency is derived from the Phase 1.2 correlation/MI matrix, per instructor note's "sensor correlation" option.


In [ ]:
print("Jobayer") 

In [ ]:
import torch
TORCH_VERSION = torch.__version__.split("+")[0]
CUDA_VERSION = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
print(f"Torch: {TORCH_VERSION}  CUDA tag: {CUDA_VERSION}")

wheel_url = f"https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_VERSION}.html"
!pip install torch-scatter torch-sparse torch-cluster -f {wheel_url} -q

!pip install torch_geometric torch_geometric_temporal -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

DATA_PATH = "/kaggle/input/datasets/akshaymp08/bhopal-agritech-dataset-for-predictive-irrigation/Bhopal-AgriTech Hybrid IoT Dataset for Predictive Irrigation.csv"
df = pd.read_csv(DATA_PATH)
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values("Timestamp").reset_index(drop=True)

print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head()

## Phase 0 — Setup & Data Understanding

In [ ]:
print(df.dtypes)
print()
print(df.columns.tolist())

In [ ]:
print(f"Start: {df['Timestamp'].min()}")
print(f"End:   {df['Timestamp'].max()}")
print(f"Total span: {df['Timestamp'].max() - df['Timestamp'].min()}")

diffs = df["Timestamp"].diff().dropna()
print("\nTop timestamp intervals:")
print(diffs.value_counts().head())

modal_interval = diffs.mode()[0]
gaps = diffs[diffs > modal_interval * 2]
print(f"\nGaps larger than 2x modal interval: {len(gaps)}")

In [ ]:
raw_features = ["Soil_Moisture", "Temperature", "Humidity"]
time_features = ["Hour", "Hour_sin", "Hour_cos"]
control_feature = "irrigation_status"
target_col = "soil_moisture_next_1m"
scaled_features = ["Soil_Moisture_scaled", "Temperature_scaled", "Humidity_scaled", "soil_moisture_next_1m_scaled"]

print("Missing values:", df.isna().sum().sum())
print()
print(df[target_col].describe())
print()
print(df[control_feature].value_counts())
print(df[control_feature].value_counts(normalize=True).round(4))

In [ ]:
for raw_col, scaled_col in [
    ("Soil_Moisture", "Soil_Moisture_scaled"),
    ("Temperature", "Temperature_scaled"),
    ("Humidity", "Humidity_scaled"),
    (target_col, "soil_moisture_next_1m_scaled"),
]:
    recomputed = (df[raw_col] - df[raw_col].min()) / (df[raw_col].max() - df[raw_col].min())
    max_diff = (recomputed - df[scaled_col]).abs().max()
    print(f"{raw_col:25s} -> {scaled_col:30s} | max abs diff: {max_diff:.6f}")

## Phase 1.1 — Descriptive Statistics & Distributions

Findings to report: Soil_Moisture/target right-skewed (skew ~0.85); 1.95% high-end outliers retained as genuine field variability; exactly **one genuine flatline event** (tolerance-based, min_run=5) found on 2026-04-03; apparent "thermal spikes" are an **hourly-resampling artifact** (all land on minute :45, confirmed via minute-of-hour distribution), not real heatwaves.

In [ ]:
stats_cols = ["Soil_Moisture", "Temperature", "Humidity", target_col]
desc = df[stats_cols].agg(["mean", "median", "std", "min", "max", "skew", "kurt"]).T
print(desc)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, stats_cols):
    df[col].plot(kind="hist", bins=50, density=True, alpha=0.6, ax=ax)
    df[col].plot(kind="kde", ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
for col in stats_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col:25s} flagged: {n_out:5d} ({n_out/len(df)*100:.2f}%)  bounds=({lower:.2f}, {upper:.2f})")

In [ ]:
# Near-flatline detection (tolerance-based, since injected freezes still have tiny jitter)
tolerance = 0.05
min_run = 5

is_near_same = df["Soil_Moisture"].diff().abs().lt(tolerance)
run_id = (~is_near_same).cumsum()
flat_runs = df.groupby(run_id).filter(lambda g: len(g) >= min_run and g["Soil_Moisture"].std() < tolerance)

print(f"Near-flat rows (tolerance={tolerance}, min_run={min_run}): {len(flat_runs)}")
if len(flat_runs) > 0:
    print(flat_runs.groupby(run_id)["Timestamp"].agg(["min", "max", "count"]))

In [ ]:
# Thermal "spike" check — confirmed to be hourly-resampling artifact, not real heatwaves
temp_diff = df["Temperature"].diff()
spike_threshold = temp_diff.std() * 3
spike_idx = temp_diff[temp_diff.abs() > spike_threshold].index

print(f"Flagged spikes: {len(spike_idx)}")
print("Minute-of-hour distribution (concentration at one value confirms resampling artifact):")
print(df.loc[spike_idx, "Timestamp"].dt.minute.value_counts().head())

In [ ]:
window = 10
rolling_std = df["Soil_Moisture"].rolling(window).std()
print(rolling_std.describe())

fig, ax = plt.subplots(figsize=(12, 4))
rolling_std.plot(ax=ax)
ax.set_title(f"Rolling {window}-min std of Soil_Moisture (noise indicator)")
plt.show()

## Phase 1.2 — Correlation / Mutual Information

This matrix is reused directly as the **Track B graph adjacency** later in this notebook. Key findings: Soil_Moisture dominates (r=0.996, trivial persistence signal); Temperature/Humidity strongly anti-correlated (r=-0.893); irrigation_status functionally noise for the target (MI≈0.0008). No site/sensor ID exists in this file, so this is a *feature* correlation structure, not a spatial adjacency.

In [ ]:
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression

feature_cols = ["Soil_Moisture", "Temperature", "Humidity", "Hour_sin", "Hour_cos", "irrigation_status"]

corr_matrix = df[feature_cols + [target_col]].corr()
print(corr_matrix.round(3))

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Pearson Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
X = df[feature_cols].values
y = df[target_col].values
mi_scores = mutual_info_regression(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=feature_cols).sort_values(ascending=False)

comparison = pd.DataFrame({
    "abs_pearson_r": corr_matrix[target_col].drop(target_col).abs(),
    "mutual_info": mi_series
}).sort_values("mutual_info", ascending=False)
print(comparison.round(4))

## Phase 1.3 — Stationarity & Autocorrelation

Findings: all series stationary at 1% level (ADF). PACF cuts off sharply after lag 1 → **AR(1)** is the justified order. ACF decays slowly despite stationarity — expected for AR(1) with φ≈0.9995–0.985, not a contradiction. Daily seasonality is visually present but SARIMA's seasonal AR term later proves statistically non-significant (p=0.76).

In [ ]:
from statsmodels.tsa.stattools import adfuller

def run_adf(series, name):
    result = adfuller(series.dropna(), autolag="AIC")
    verdict = "STATIONARY" if result[1] < 0.05 else "NON-STATIONARY"
    print(f"{name}: ADF={result[0]:.4f}  p={result[1]:.6f}  -> {verdict}")
    return result[1]

run_adf(df["Soil_Moisture"], "Soil_Moisture")
run_adf(df["Temperature"], "Temperature")
run_adf(df["Humidity"], "Humidity")

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df["Soil_Moisture"].dropna(), lags=60, ax=axes[0])
axes[0].set_title("ACF — Soil_Moisture")
plot_pacf(df["Soil_Moisture"].dropna(), lags=60, ax=axes[1], method="ywm")
axes[1].set_title("PACF — Soil_Moisture")
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(df["Soil_Moisture"], model="additive", period=1440, extrapolate_trend="freq")
fig = decomposition.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.show()

## Phase 1.5 — Hypothesis Testing (design)

Executed in Phase 4 with rolling-origin CV fold-wise errors:
- **H1**: mean RMSE does not differ between statistical baselines and sequence deep learning models (paired t-test / Wilcoxon).
- **H2**: mean RMSE does not differ between ARIMA+exogenous and true SARIMA seasonal modeling.

`results_df` below accumulates single-holdout results for a quick running summary; `cv_results_df` (Phase 4) holds the proper fold-wise results used for the actual hypothesis tests.

In [ ]:
results_columns = ["fold", "model", "rmse", "mae", "r2"]
results_df = pd.DataFrame(columns=results_columns)

split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()
print(f"Train: {len(train_df):,} rows | Test: {len(test_df):,} rows")

## Phase 2 — Baseline Statistical Models

Persistence, ARIMA(1,0,0)+exogenous Fourier features (Hour_sin/cos), Holt-Winters, and SARIMA(1,0,0)(1,0,0,24) fit on an **hourly-resampled** copy of the series (true 1-minute SARIMA at s=1440 was computationally infeasible — confirmed by a stalled fit during development; hourly resampling with s=24 keeps the seasonal test tractable, with the resolution mismatch flagged explicitly wherever SARIMA results are compared).

All models below use a **one-step-ahead evaluation** (`.filter()` + `get_prediction(dynamic=False)`), not a blind multi-step `.forecast()` — the latter was tested during development and produced a catastrophic, misleading RMSE because it never sees true intervening observations, which does not match this task's actual real-time-forecasting setup.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def eval_metrics(y_true, y_pred):
    return (
        np.sqrt(mean_squared_error(y_true, y_pred)),
        mean_absolute_error(y_true, y_pred),
        r2_score(y_true, y_pred),
    )

# --- Persistence baseline ---
y_true = test_df[target_col].values
persistence_pred = test_df["Soil_Moisture"].values
rmse, mae, r2 = eval_metrics(y_true, persistence_pred)
print(f"Persistence — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}")
results_df = pd.concat([results_df, pd.DataFrame([{"fold": "holdout", "model": "Persistence", "rmse": rmse, "mae": mae, "r2": r2}])], ignore_index=True)

In [ ]:
# ============================================================
# Extended metrics: MAPE + Training Time + Parameter Count
# (Section 7.1 requires MAPE for forecasting groups; Section 4.4
#  requires training time and model size in the comparison)
# ============================================================
import time

def mean_absolute_percentage_error_safe(y_true, y_pred, epsilon=1e-6):
    """Safe MAPE — avoids divide-by-near-zero blowups."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), epsilon)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100)

def count_params(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))

def log_result(results_df, name, y_true, y_pred, fold="holdout",
                train_time=None, model=None):
    """Drop-in replacement for the eval_metrics + pd.concat pattern used
    throughout this notebook — computes RMSE/MAE/MAPE/R2 and appends a row
    with training_time_s and num_params alongside them."""
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    mape = mean_absolute_percentage_error_safe(y_true, y_pred)
    r2 = float(r2_score(y_true, y_pred))
    n_params = count_params(model) if model is not None else np.nan
    row = {
        "fold": fold, "model": name,
        "rmse": rmse, "mae": mae, "mape": mape, "r2": r2,
        "training_time_s": train_time, "num_params": n_params,
    }
    extra = f"  | {train_time:.1f}s, {n_params:,} params" if (train_time is not None and model is not None) else ""
    print(f"{name} — RMSE: {rmse:.4f}  MAE: {mae:.4f}  MAPE: {mape:.2f}%  R2: {r2:.4f}{extra}")
    return pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

exog_cols = ["Hour_sin", "Hour_cos"]

arima_model = SARIMAX(train_df["Soil_Moisture"], exog=train_df[exog_cols],
                       order=(1, 0, 0), enforce_stationarity=False, enforce_invertibility=False)
arima_fit = arima_model.fit(disp=False)
print(arima_fit.summary())

In [ ]:
# One-step-ahead evaluation (correct method — do not use .forecast() for a walk-forward task)
full_arima = SARIMAX(df["Soil_Moisture"], exog=df[exog_cols],
                      order=(1, 0, 0), enforce_stationarity=False, enforce_invertibility=False)
full_arima_fit = full_arima.filter(arima_fit.params)

pred = full_arima_fit.get_prediction(start=split_idx, end=len(df) - 1, dynamic=False)
arima_pred = pred.predicted_mean.values

rmse, mae, r2 = eval_metrics(y_true, arima_pred)
print(f"ARIMA(1,0,0)+exog — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}")
results_df = pd.concat([results_df, pd.DataFrame([{"fold": "holdout", "model": "ARIMA_exog", "rmse": rmse, "mae": mae, "r2": r2}])], ignore_index=True)

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

full_hw_model = ExponentialSmoothing(df["Soil_Moisture"], trend="add", seasonal=None)
full_hw_fit = full_hw_model.fit()
hw_pred = full_hw_fit.fittedvalues.iloc[split_idx:].values

rmse, mae, r2 = eval_metrics(y_true, hw_pred)
print(f"Holt-Winters — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}")
print(f"(Note: smoothing_trend beta may optimize to ~0 — effectively simple exp. smoothing, not a real trend component.)")
results_df = pd.concat([results_df, pd.DataFrame([{"fold": "holdout", "model": "Holt-Winters", "rmse": rmse, "mae": mae, "r2": r2}])], ignore_index=True)

In [ ]:
# SARIMA on hourly-resampled series (s=1440 at 1-min resolution was infeasible)
hourly_train = train_df.set_index("Timestamp")["Soil_Moisture"].resample("1h").mean()
hourly_test = test_df.set_index("Timestamp")["Soil_Moisture"].resample("1h").mean()
full_hourly = pd.concat([hourly_train, hourly_test])

sarima_model = SARIMAX(hourly_train, order=(1, 0, 0), seasonal_order=(1, 0, 0, 24),
                        enforce_stationarity=False, enforce_invertibility=False)
sarima_fit = sarima_model.fit(disp=False)
print(sarima_fit.summary())

In [ ]:
full_sarima = SARIMAX(full_hourly, order=(1, 0, 0), seasonal_order=(1, 0, 0, 24),
                      enforce_stationarity=False, enforce_invertibility=False)
full_sarima_fit = full_sarima.filter(sarima_fit.params)

pred = full_sarima_fit.get_prediction(start=len(hourly_train), end=len(full_hourly) - 1, dynamic=False)
sarima_pred = pred.predicted_mean.values
y_true_hourly = hourly_test.values

rmse, mae, r2 = eval_metrics(y_true_hourly, sarima_pred)
print(f"SARIMA(1,0,0)(1,0,0,24) [HOURLY — not directly comparable to 1-min models] — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}")
# Note: seasonal AR term (ar.S.L24) tests non-significant (p~0.76) — daily persistence, not
# genuine seasonal structure, is doing the work here. See report Discussion section.

plt.figure(figsize=(14, 5))
plt.plot(hourly_test.index, y_true_hourly, label="Actual", alpha=0.8)
plt.plot(hourly_test.index, sarima_pred, label="SARIMA predicted", alpha=0.8)
plt.title("SARIMA — Actual vs Predicted (hourly)")
plt.legend()
plt.tight_layout()
plt.show()

## Phase 3 — Sequence Deep Learning Models (LSTM / GRU)

60-minute lookback window, chosen from Phase 1.3's PACF/ACF behavior. Features are re-scaled here (train-fit only) rather than using the stale provided `_scaled` columns.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import MinMaxScaler

print("GPU available:", tf.config.list_physical_devices('GPU'))
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

SEQ_LENGTH = 60

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
train_X_scaled = scaler_X.fit_transform(train_df[feature_cols])
test_X_scaled = scaler_X.transform(test_df[feature_cols])
train_y_scaled = scaler_y.fit_transform(train_df[[target_col]])
test_y_scaled = scaler_y.transform(test_df[[target_col]])

def create_sequences(X, y, seq_length):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i + seq_length])
        y_seq.append(y[i + seq_length])
    return np.array(X_seq), np.array(y_seq)

X_train_seq, y_train_seq = create_sequences(train_X_scaled, train_y_scaled, SEQ_LENGTH)
X_test_seq, y_test_seq = create_sequences(test_X_scaled, test_y_scaled, SEQ_LENGTH)
print(f"X_train_seq: {X_train_seq.shape}  X_test_seq: {X_test_seq.shape}")

In [ ]:
tf.random.set_seed(42)
n_features = X_train_seq.shape[2]

lstm_model = models.Sequential([
    layers.Input(shape=(SEQ_LENGTH, n_features)),
    layers.LSTM(32),
    layers.Dense(16, activation="relu"),
    layers.Dense(1),
])
lstm_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
history = lstm_model.fit(X_train_seq, y_train_seq, validation_split=0.1, epochs=15, batch_size=128, verbose=1)

In [ ]:
lstm_pred_scaled = lstm_model.predict(X_test_seq)
lstm_pred = scaler_y.inverse_transform(lstm_pred_scaled).flatten()
y_true_seq = scaler_y.inverse_transform(y_test_seq).flatten()

rmse, mae, r2 = eval_metrics(y_true_seq, lstm_pred)
print(f"LSTM — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}")
results_df = pd.concat([results_df, pd.DataFrame([{"fold": "holdout", "model": "LSTM", "rmse": rmse, "mae": mae, "r2": r2}])], ignore_index=True)

In [ ]:
gru_model = models.Sequential([
    layers.Input(shape=(SEQ_LENGTH, n_features)),
    layers.GRU(32),
    layers.Dense(16, activation="relu"),
    layers.Dense(1),
])
gru_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
gru_history = gru_model.fit(X_train_seq, y_train_seq, validation_split=0.1, epochs=20, batch_size=128, verbose=1)

In [ ]:
gru_pred_scaled = gru_model.predict(X_test_seq)
gru_pred = scaler_y.inverse_transform(gru_pred_scaled).flatten()

rmse, mae, r2 = eval_metrics(y_true_seq, gru_pred)
print(f"GRU — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}")
results_df = pd.concat([results_df, pd.DataFrame([{"fold": "holdout", "model": "GRU", "rmse": rmse, "mae": mae, "r2": r2}])], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train"); axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("LSTM Loss"); axes[0].legend()
axes[1].plot(gru_history.history["loss"], label="train"); axes[1].plot(gru_history.history["val_loss"], label="val")
axes[1].set_title("GRU Loss"); axes[1].legend()
plt.tight_layout()
plt.show()

print(results_df)

## Phase 4 — Rolling-Origin Cross-Validation + Hypothesis Testing

5 expanding-window folds, refitting every 1-minute-resolution model per fold (SARIMA excluded — different resolution, already caveated). This gives the fold-wise error distributions needed for genuine paired hypothesis tests (a single holdout split cannot support a paired t-test).

In [ ]:
N = len(df)
n_blocks = 10
block_size = N // n_blocks
boundaries = [block_size * i for i in range(1, n_blocks + 1)]
boundaries[-1] = N

fold_splits = [(0, boundaries[i], boundaries[i+1]) for i in range(len(boundaries) - 1)]
for i, (s, tr, te) in enumerate(fold_splits, 1):
    print(f"Fold {i}: train 0-{tr} ({tr}) | test {tr}-{te} ({te-tr})")

In [ ]:
def fit_arima_exog_fold(train_slice, full_slice, train_len):
    model = SARIMAX(train_slice["Soil_Moisture"], exog=train_slice[exog_cols],
                     order=(1, 0, 0), enforce_stationarity=False, enforce_invertibility=False)
    fit = model.fit(disp=False)
    full_model = SARIMAX(full_slice["Soil_Moisture"], exog=full_slice[exog_cols],
                          order=(1, 0, 0), enforce_stationarity=False, enforce_invertibility=False)
    full_fit = full_model.filter(fit.params)
    pred = full_fit.get_prediction(start=train_len, end=len(full_slice) - 1, dynamic=False)
    return pred.predicted_mean.values

def fit_hw_fold(full_slice, train_len):
    model = ExponentialSmoothing(full_slice["Soil_Moisture"], trend="add", seasonal=None)
    fit = model.fit()
    return fit.fittedvalues.iloc[train_len:].values

def fit_lstm_gru_fold(train_slice, test_slice, cell_type="LSTM", epochs=10):
    sX, sy = MinMaxScaler(), MinMaxScaler()
    trX = sX.fit_transform(train_slice[feature_cols]); teX = sX.transform(test_slice[feature_cols])
    trY = sy.fit_transform(train_slice[[target_col]]); teY = sy.transform(test_slice[[target_col]])
    Xtr, ytr = create_sequences(trX, trY, SEQ_LENGTH)
    Xte, yte = create_sequences(teX, teY, SEQ_LENGTH)
    layer = layers.LSTM(32) if cell_type == "LSTM" else layers.GRU(32)
    model = models.Sequential([layers.Input(shape=(SEQ_LENGTH, len(feature_cols))), layer,
                                layers.Dense(16, activation="relu"), layers.Dense(1)])
    model.compile(optimizer="adam", loss="mse")
    model.fit(Xtr, ytr, epochs=epochs, batch_size=128, verbose=0)
    pred_scaled = model.predict(Xte, verbose=0)
    pred = sy.inverse_transform(pred_scaled).flatten()
    yt = sy.inverse_transform(yte).flatten()
    return yt, pred

print("Fold helper functions defined.")

In [ ]:
fold_results = []
for fold_num, (start, train_end, test_end) in enumerate(fold_splits, 1):
    print(f"=== Fold {fold_num} ===")
    train_slice = df.iloc[start:train_end].reset_index(drop=True)
    test_slice = df.iloc[train_end:test_end].reset_index(drop=True)
    full_slice = df.iloc[start:test_end].reset_index(drop=True)
    train_len = len(train_slice)
    yt = test_slice[target_col].values

    p_pred = test_slice["Soil_Moisture"].values
    r, m, r2_ = eval_metrics(yt, p_pred)
    fold_results.append({"fold": fold_num, "model": "Persistence", "rmse": r, "mae": m, "r2": r2_})

    a_pred = fit_arima_exog_fold(train_slice, full_slice, train_len)
    r, m, r2_ = eval_metrics(yt, a_pred)
    fold_results.append({"fold": fold_num, "model": "ARIMA_exog", "rmse": r, "mae": m, "r2": r2_})

    hw_pred = fit_hw_fold(full_slice, train_len)
    r, m, r2_ = eval_metrics(yt, hw_pred)
    fold_results.append({"fold": fold_num, "model": "Holt-Winters", "rmse": r, "mae": m, "r2": r2_})

    ytl, lstm_p = fit_lstm_gru_fold(train_slice, test_slice, "LSTM")
    r, m, r2_ = eval_metrics(ytl, lstm_p)
    fold_results.append({"fold": fold_num, "model": "LSTM", "rmse": r, "mae": m, "r2": r2_})

    ytg, gru_p = fit_lstm_gru_fold(train_slice, test_slice, "GRU")
    r, m, r2_ = eval_metrics(ytg, gru_p)
    fold_results.append({"fold": fold_num, "model": "GRU", "rmse": r, "mae": m, "r2": r2_})
    print(f"Fold {fold_num} done.")

# cv_results_df = pd.DataFrame(fold_results)
# print(cv_results_df.pivot(index="fold", columns="model", values="rmse").round(4))


cv_results_df = pd.DataFrame(fold_results)

metrics_to_show = ["rmse", "mae", "r2"]

# One pivot table per metric — fold x model, matching your original RMSE-only version
for metric in metrics_to_show:
    print(f"\n=== {metric.upper()} by fold ===")
    pivot = cv_results_df.pivot(index="fold", columns="model", values=metric).round(4)
    display(pivot)

# Mean/std across folds per model per metric — this is what your H1/H2 hypothesis
# tests are actually built on, so useful to see side by side with the raw folds
print("\n=== Summary across folds (mean, std) ===")
summary = cv_results_df.groupby("model")[metrics_to_show].agg(["mean", "std"]).round(4)
display(summary)

# Compact "mean ± std" version — this is the format that drops straight into
print("\n=== Compact summary (report-ready) ===")
compact = pd.DataFrame(index=cv_results_df["model"].unique())
for metric in metrics_to_show:
    g = cv_results_df.groupby("model")[metric].agg(["mean", "std"])
    compact[metric.upper()] = g.apply(lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1)
display(compact)

In [ ]:
from scipy.stats import ttest_rel, wilcoxon

def compare_models(df_cv, model_a, model_b, metric="rmse"):
    a = df_cv[df_cv.model == model_a].sort_values("fold")[metric].values
    b = df_cv[df_cv.model == model_b].sort_values("fold")[metric].values
    t_stat, t_p = ttest_rel(a, b)
    try:
        w_stat, w_p = wilcoxon(a, b)
    except ValueError:
        w_stat, w_p = np.nan, np.nan
    print(f"{model_a} vs {model_b}: mean {a.mean():.4f} vs {b.mean():.4f} | t p={t_p:.4f} | Wilcoxon p={w_p:.4f}")

compare_models(cv_results_df, "ARIMA_exog", "LSTM")
compare_models(cv_results_df, "ARIMA_exog", "GRU")
compare_models(cv_results_df, "LSTM", "GRU")
compare_models(cv_results_df, "Persistence", "LSTM")
compare_models(cv_results_df, "Persistence", "ARIMA_exog")



## Track B — Graph-Based Modeling (GNN)

**Graph definition** (per instructor note — model soil moisture, temperature, humidity, irrigation status, and cyclic time features as a temporal sensor graph):
- **Nodes (6)**: Soil_Moisture, Temperature, Humidity, irrigation_status, Hour_sin, Hour_cos
- **Edges**: derived from the Phase 1.2 correlation matrix (|Pearson r| as edge weight, fully connected) — this is the "sensor correlation" adjacency option, appropriate since this dataset has no spatial/sensor ID to support a spatial graph
- **Node features**: 60-minute sliding windows per node (matches Track A's SEQ_LENGTH for direct comparability)
- **Temporal construction**: fixed topology (global correlation, doesn't change over time) + evolving per-window node features — the standard assumption behind T-GCN/STGCN
- **Architectures covered** — static: GCN, GAT, GraphSAGE, GIN; temporal/spatio-temporal: T-GCN, DCRNN, A3T-GCN, STGCN, LSTM-GNN hybrid, and a simplified DySAT-inspired model (full list from the instructor note)
- **Caveat to report honestly**: with only 6 nodes, this is a far smaller graph than typical GNN benchmarks; most modeling power comes from the temporal component, not graph structure. A few of these architectures (LSTM-GNN hybrid, DySAT-inspired) use nested per-sample loops that are computationally expensive on this setup, so they are trained/evaluated on a subsample — flagged explicitly wherever this applies, and not to be compared directly against full-test-set results without noting it.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

graph_nodes = ["Soil_Moisture", "Temperature", "Humidity", "irrigation_status", "Hour_sin", "Hour_cos"]
n_nodes = len(graph_nodes)

corr_graph = df[graph_nodes].corr().values
np.fill_diagonal(corr_graph, 0)

edge_index_list, edge_weight_list = [], []
for i in range(n_nodes):
    for j in range(n_nodes):
        if i != j:
            edge_index_list.append([i, j])
            edge_weight_list.append(abs(corr_graph[i, j]))

edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
edge_weight = torch.tensor(edge_weight_list, dtype=torch.float)
edge_index_dev = edge_index.to(device)
edge_weight_dev = edge_weight.to(device)

print("Adjacency (|correlation|):")
print(pd.DataFrame(corr_graph, index=graph_nodes, columns=graph_nodes).abs().round(3))

In [ ]:
# Track B Graph Visualization 

import networkx as nx
import os

os.makedirs("/kaggle/working/outputs", exist_ok=True)

G = nx.Graph()
for i, name in enumerate(graph_nodes):
    G.add_node(i, label=name)
for i in range(n_nodes):
    for j in range(i + 1, n_nodes):
        G.add_edge(i, j, weight=abs(corr_graph[i, j]))

display_labels = {i: n.replace("_", "\n") for i, n in enumerate(graph_nodes)}
pos = nx.circular_layout(G, scale=1.0)
weights = [G[u][v]["weight"] for u, v in G.edges()]
edge_widths = [1 + w * 6 for w in weights]

fig, ax = plt.subplots(figsize=(8, 8))
nx.draw_networkx_nodes(G, pos, node_size=3000, node_color="#4C72B0",
                        edgecolors="black", linewidths=1.2, ax=ax)
nx.draw_networkx_labels(G, pos, labels=display_labels, font_size=7.5,
                         font_color="white", ax=ax)
nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color=weights,
                        edge_cmap=plt.cm.viridis, alpha=0.85, ax=ax)
edge_labels = {(u, v): f"{G[u][v]['weight']:.2f}" for u, v in G.edges()}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7, ax=ax)

ax.set_title("Track B Graph Structure — Variable/Channel Graph\n"
              "(edge width & color = |Pearson r|, fully connected, 6 nodes)")
ax.axis("off")
xs, ys = [p[0] for p in pos.values()], [p[1] for p in pos.values()]
pad = 0.4
ax.set_xlim(min(xs) - pad, max(xs) + pad)
ax.set_ylim(min(ys) - pad, max(ys) + pad)
plt.tight_layout()
plt.savefig("/kaggle/working/outputs/graph_structure.png", dpi=150, bbox_inches="tight")
plt.show()

strongest = max(G.edges(data=True), key=lambda e: e[2]["weight"])
weakest = min(G.edges(data=True), key=lambda e: e[2]["weight"])
print(f"Nodes: {n_nodes} | Undirected edges shown: {G.number_of_edges()} "
      f"(models use all {edge_index.shape[1]} directed edges, i.e. both directions of each pair)")
print(f"Strongest connection: {graph_nodes[strongest[0]]} — {graph_nodes[strongest[1]]}  "
      f"(|r| = {strongest[2]['weight']:.3f})")
print(f"Weakest connection:   {graph_nodes[weakest[0]]} — {graph_nodes[weakest[1]]}  "
      f"(|r| = {weakest[2]['weight']:.3f})")

In [ ]:
WINDOW = 60

node_scalers = {}
train_scaled_g, test_scaled_g = {}, {}
for col in graph_nodes:
    sc = MinMaxScaler()
    train_scaled_g[col] = sc.fit_transform(train_df[[col]]).flatten()
    test_scaled_g[col] = sc.transform(test_df[[col]]).flatten()
    node_scalers[col] = sc

target_scaler_g = MinMaxScaler()
train_target_scaled_g = target_scaler_g.fit_transform(train_df[[target_col]]).flatten()
test_target_scaled_g = target_scaler_g.transform(test_df[[target_col]]).flatten()

def build_windows(scaled_dict, target_scaled, window):
    n = len(next(iter(scaled_dict.values())))
    X, y = [], []
    for t in range(n - window):
        snapshot = np.stack([scaled_dict[col][t:t + window] for col in graph_nodes])
        X.append(snapshot)
        y.append(target_scaled[t + window])
    return np.array(X), np.array(y)

X_train_g, y_train_g = build_windows(train_scaled_g, train_target_scaled_g, WINDOW)
X_test_g, y_test_g = build_windows(test_scaled_g, test_target_scaled_g, WINDOW)
print(f"X_train_g: {X_train_g.shape}  X_test_g: {X_test_g.shape}")

In [ ]:
X_train_t = torch.tensor(X_train_g, dtype=torch.float32)
y_train_t = torch.tensor(y_train_g, dtype=torch.float32).unsqueeze(-1)
batch_size = 128
n_batches = len(X_train_t) // batch_size
X_test_t = torch.tensor(X_test_g, dtype=torch.float32)
y_true_g = target_scaler_g.inverse_transform(y_test_g.reshape(-1, 1)).flatten()

print("All ok")

In [ ]:
from torch_geometric.nn import GATConv

class GATForecaster(nn.Module):
    def __init__(self, in_channels, hidden_channels=32, heads=2):
        super().__init__()
        self.gat1 = GATConv(in_channels, hidden_channels, heads=heads, edge_dim=1)
        self.gat2 = GATConv(hidden_channels * heads, hidden_channels, heads=1, edge_dim=1)
        self.readout = nn.Linear(hidden_channels * n_nodes, 1)

    def forward(self, x, edge_index, edge_weight):
        edge_attr = edge_weight.unsqueeze(-1)
        outs = []
        for b in range(x.shape[0]):
            h = F.elu(self.gat1(x[b], edge_index, edge_attr))
            h = F.elu(self.gat2(h, edge_index, edge_attr))
            outs.append(h.flatten())
        return self.readout(torch.stack(outs))


gat_model = GATForecaster(in_channels=WINDOW).to(device)
edge_index_dev = edge_index.to(device)
edge_weight_dev = edge_weight.to(device)
optimizer = torch.optim.Adam(gat_model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()


In [ ]:

train_start = time.time()
gat_model.train()
for epoch in range(5):
    epoch_loss = 0.0
    perm = torch.randperm(len(X_train_t))
    for i in range(n_batches):
        idx = perm[i*batch_size:(i+1)*batch_size]
        xb, yb = X_train_t[idx].to(device), y_train_t[idx].to(device)
        optimizer.zero_grad()
        pred = gat_model(xb, edge_index_dev, edge_weight_dev)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={epoch_loss/n_batches:.6f}")

gat_train_time = time.time() - train_start

In [ ]:
gat_model.eval()
X_test_t = torch.tensor(X_test_g, dtype=torch.float32)
preds = []
with torch.no_grad():
    for i in range(0, len(X_test_t), batch_size):
        xb = X_test_t[i:i+batch_size].to(device)
        preds.append(gat_model(xb, edge_index_dev, edge_weight_dev).cpu().numpy())
gat_pred_scaled = np.concatenate(preds).flatten()
gat_pred = target_scaler_g.inverse_transform(gat_pred_scaled.reshape(-1, 1)).flatten()
y_true_g = target_scaler_g.inverse_transform(y_test_g.reshape(-1, 1)).flatten()

rmse, mae, r2 = eval_metrics(y_true_g, gat_pred)
print(f"GAT — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}")
results_df = log_result(results_df, "GAT", y_true_g, gat_pred, train_time=gat_train_time, model=gat_model)

# T-GCN

In [ ]:
from torch_geometric_temporal.nn.recurrent import TGCN

class TGCNForecaster(nn.Module):
    def __init__(self, in_channels=1, hidden_channels=32):
        super().__init__()
        self.tgcn = TGCN(in_channels, hidden_channels)
        self.readout = nn.Linear(hidden_channels * n_nodes, 1)

    def forward(self, x, edge_index, edge_weight, h=None):
        h = self.tgcn(x, edge_index, edge_weight, h)
        return self.readout(h.flatten()), h

tgcn_model = TGCNForecaster().to(device)
optimizer_t = torch.optim.Adam(tgcn_model.parameters(), lr=1e-3)

train_seq = np.stack([train_scaled_g[col] for col in graph_nodes], axis=1)
CHUNK = 60
n_chunks = (len(train_seq) - 1) // CHUNK


train_start = time.time()

tgcn_model.train()
for epoch in range(5):
    h = None
    total_loss = 0.0
    for c in range(n_chunks):
        optimizer_t.zero_grad()
        chunk_loss = 0.0
        h_local = h.detach() if h is not None else None
        for t in range(c*CHUNK, (c+1)*CHUNK):
            x_t = torch.tensor(train_seq[t], dtype=torch.float32).unsqueeze(-1).to(device)
            y_t = torch.tensor([train_target_scaled_g[t]], dtype=torch.float32).to(device)
            pred, h_local = tgcn_model(x_t, edge_index_dev, edge_weight_dev, h_local)
            chunk_loss = chunk_loss + loss_fn(pred, y_t)
        chunk_loss = chunk_loss / CHUNK
        chunk_loss.backward()
        optimizer_t.step()
        h = h_local.detach()
        total_loss += chunk_loss.item()
    print(f"Epoch {epoch+1}: avg chunk loss={total_loss/n_chunks:.6f}")

tgcn_train_time = time.time() - train_start

In [ ]:
test_seq = np.stack([test_scaled_g[col] for col in graph_nodes], axis=1)
tgcn_model.eval()
tgcn_preds = []
h = None
with torch.no_grad():
    for t in range(len(test_seq)):
        x_t = torch.tensor(test_seq[t], dtype=torch.float32).unsqueeze(-1).to(device)
        pred, h = tgcn_model(x_t, edge_index_dev, edge_weight_dev, h)
        tgcn_preds.append(pred.cpu().item())

tgcn_pred_scaled = np.array(tgcn_preds[:-1])
y_true_tgcn_scaled = test_target_scaled_g[1:]
tgcn_pred = target_scaler_g.inverse_transform(tgcn_pred_scaled.reshape(-1, 1)).flatten()
y_true_tgcn = target_scaler_g.inverse_transform(y_true_tgcn_scaled.reshape(-1, 1)).flatten()

rmse, mae, r2 = eval_metrics(y_true_tgcn, tgcn_pred)
print(f"T-GCN — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}")
results_df = log_result(results_df, "T-GCN", y_true_tgcn, tgcn_pred, train_time=tgcn_train_time, model=tgcn_model)

### Additional Static-Graph Architectures — GCN, GraphSAGE, GIN

Per instructor note to cover the full architecture list. These reuse the same windowed-snapshot input (`X_train_g`/`X_test_g`) and training loop pattern as GAT above, so results are directly comparable. GraphSAGE and GIN's standard PyG layers don't accept edge weights natively (they aggregate over unweighted neighborhoods), so the correlation-derived edge weights only inform GCN and GAT directly — worth noting as a limitation when comparing across architectures.

# GCN

In [ ]:
from torch_geometric.nn import GCNConv

class GCNForecaster(nn.Module):
    def __init__(self, in_channels, hidden_channels=32):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.readout = nn.Linear(hidden_channels * n_nodes, 1)

    def forward(self, x, edge_index, edge_weight):
        outs = []
        for b in range(x.shape[0]):
            h = F.relu(self.conv1(x[b], edge_index, edge_weight))
            h = F.relu(self.conv2(h, edge_index, edge_weight))
            outs.append(h.flatten())
        return self.readout(torch.stack(outs))

def train_gcn(epochs=1):
    model = GCNForecaster(in_channels=WINDOW).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        perm = torch.randperm(len(X_train_t))
        for i in range(n_batches):
            idx = perm[i*batch_size:(i+1)*batch_size]
            xb, yb = X_train_t[idx].to(device), y_train_t[idx].to(device)
            optimizer.zero_grad()
            pred = model(xb, edge_index_dev, edge_weight_dev)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        print(f"  [GCN] Epoch {epoch+1}: loss={epoch_loss/n_batches:.6f}")
    return model

def eval_gcn(model):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_test_t), batch_size):
            xb = X_test_t[i:i+batch_size].to(device)
            preds.append(model(xb, edge_index_dev, edge_weight_dev).cpu().numpy())
    pred_scaled = np.concatenate(preds).flatten()
    return target_scaler_g.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()

print("=== Training GCN ===")
t0 = time.time()
gcn_model = train_gcn(epochs=5)
gcn_train_time = time.time() - t0
gcn_pred = eval_gcn(gcn_model)
results_df = log_result(results_df, "GCN", y_true_g, gcn_pred, train_time=gcn_train_time, model=gcn_model)

# GraphSAGE

In [ ]:
from torch_geometric.nn import SAGEConv

class GraphSAGEForecaster(nn.Module):
    def __init__(self, in_channels, hidden_channels=32):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.readout = nn.Linear(hidden_channels * n_nodes, 1)

    def forward(self, x, edge_index):
        outs = []
        for b in range(x.shape[0]):
            h = F.relu(self.conv1(x[b], edge_index))
            h = F.relu(self.conv2(h, edge_index))
            outs.append(h.flatten())
        return self.readout(torch.stack(outs))

def train_sage(epochs=15):
    model = GraphSAGEForecaster(in_channels=WINDOW).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        perm = torch.randperm(len(X_train_t))
        for i in range(n_batches):
            idx = perm[i*batch_size:(i+1)*batch_size]
            xb, yb = X_train_t[idx].to(device), y_train_t[idx].to(device)
            optimizer.zero_grad()
            pred = model(xb, edge_index_dev)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        print(f"  [GraphSAGE] Epoch {epoch+1}: loss={epoch_loss/n_batches:.6f}")
    return model

def eval_sage(model):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_test_t), batch_size):
            xb = X_test_t[i:i+batch_size].to(device)
            preds.append(model(xb, edge_index_dev).cpu().numpy())
    pred_scaled = np.concatenate(preds).flatten()
    return target_scaler_g.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()

print("=== Training GraphSAGE ===")
t0 = time.time()
sage_model = train_sage(epochs=5)
sage_train_time = time.time() - t0
sage_pred = eval_sage(sage_model)
results_df = log_result(results_df, "GraphSAGE", y_true_g, sage_pred, train_time=sage_train_time, model=sage_model)

# GIN

In [ ]:
from torch_geometric.nn import GINConv


edge_index_dev = edge_index.to(device)
edge_weight_dev = edge_weight.to(device)
optimizer = torch.optim.Adam(gat_model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

class GINForecaster(nn.Module):
    def __init__(self, in_channels, hidden_channels=32):
        super().__init__()
        nn1 = nn.Sequential(nn.Linear(in_channels, hidden_channels), nn.ReLU(),
                             nn.LayerNorm(hidden_channels), nn.Linear(hidden_channels, hidden_channels))
        nn2 = nn.Sequential(nn.Linear(hidden_channels, hidden_channels), nn.ReLU(),
                             nn.LayerNorm(hidden_channels), nn.Linear(hidden_channels, hidden_channels))
        self.conv1 = GINConv(nn1, train_eps=True)
        self.conv2 = GINConv(nn2, train_eps=True)
        self.readout = nn.Linear(hidden_channels * n_nodes, 1)

    def forward(self, x, edge_index):
        outs = []
        for b in range(x.shape[0]):
            h = F.relu(self.conv1(x[b], edge_index))
            h = F.relu(self.conv2(h, edge_index))
            outs.append(h.flatten())
        return self.readout(torch.stack(outs))

def train_gin(epochs):
    model = GINForecaster(in_channels=WINDOW).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)  # lower LR than GCN/SAGE
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        perm = torch.randperm(len(X_train_t))
        for i in range(n_batches):
            idx = perm[i*batch_size:(i+1)*batch_size]
            xb, yb = X_train_t[idx].to(device), y_train_t[idx].to(device)
            optimizer.zero_grad()
            pred = model(xb, edge_index_dev)
            loss = loss_fn(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # key fix
            optimizer.step()
            epoch_loss += loss.item()
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  [GIN] Epoch {epoch+1}/{epochs}: loss={epoch_loss/n_batches:.6f}")
    return model

def eval_gin(model):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X_test_t), batch_size):
            xb = X_test_t[i:i+batch_size].to(device)
            preds.append(model(xb, edge_index_dev).cpu().numpy())
    pred_scaled = np.concatenate(preds).flatten()
    return target_scaler_g.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()

print("=== Training GIN (stabilized) ===")
t0 = time.time()
gin_model = train_gin(epochs=5)
gin_train_time = time.time() - t0
gin_pred = eval_gin(gin_model)
results_df = log_result(results_df, "GIN", y_true_g, gin_pred, train_time=gin_train_time, model=gin_model)

# DCRNN

### Additional Temporal Architecture — DCRNN

Same recurrent, step-by-step training pattern as T-GCN (shares the same call signature: `(x, edge_index, edge_weight, H)`), so this is a direct architectural swap — diffusion convolution instead of T-GCN's GRU-style gating.

In [ ]:
from torch_geometric_temporal.nn.recurrent import DCRNN

class DCRNNForecaster(nn.Module):
    def __init__(self, in_channels=1, hidden_channels=32, K=2):
        super().__init__()
        self.dcrnn = DCRNN(in_channels, hidden_channels, K=K)
        self.readout = nn.Linear(hidden_channels * n_nodes, 1)

    def forward(self, x, edge_index, edge_weight, h=None):
        h = self.dcrnn(x, edge_index, edge_weight, h)
        return self.readout(h.flatten()), h

dcrnn_model = DCRNNForecaster().to(device)
optimizer_d = torch.optim.Adam(dcrnn_model.parameters(), lr=1e-3)


train_start = time.time() 
dcrnn_model.train()
for epoch in range(5):
    h = None
    total_loss = 0.0
    for c in range(n_chunks):
        optimizer_d.zero_grad()
        chunk_loss = 0.0
        h_local = h.detach() if h is not None else None
        for t in range(c*CHUNK, (c+1)*CHUNK):
            x_t = torch.tensor(train_seq[t], dtype=torch.float32).unsqueeze(-1).to(device)
            y_t = torch.tensor([train_target_scaled_g[t]], dtype=torch.float32).to(device)
            pred, h_local = dcrnn_model(x_t, edge_index_dev, edge_weight_dev, h_local)
            chunk_loss = chunk_loss + loss_fn(pred, y_t)
        chunk_loss = chunk_loss / CHUNK
        chunk_loss.backward()
        optimizer_d.step()
        h = h_local.detach()
        total_loss += chunk_loss.item()
    print(f"Epoch {epoch+1}: avg chunk loss={total_loss/n_chunks:.6f}")

dcrnn_train_time = time.time() - train_start

In [ ]:
dcrnn_model.eval()
dcrnn_preds = []
h = None
with torch.no_grad():
    for t in range(len(test_seq)):
        x_t = torch.tensor(test_seq[t], dtype=torch.float32).unsqueeze(-1).to(device)
        pred, h = dcrnn_model(x_t, edge_index_dev, edge_weight_dev, h)
        dcrnn_preds.append(pred.cpu().item())

dcrnn_pred_scaled = np.array(dcrnn_preds[:-1])
y_true_dcrnn_scaled = test_target_scaled_g[1:]
dcrnn_pred = target_scaler_g.inverse_transform(dcrnn_pred_scaled.reshape(-1, 1)).flatten()
y_true_dcrnn = target_scaler_g.inverse_transform(y_true_dcrnn_scaled.reshape(-1, 1)).flatten()

rmse, mae, r2 = eval_metrics(y_true_dcrnn, dcrnn_pred)
print(f"DCRNN — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}")
results_df = log_result(results_df, "DCRNN", y_true_dcrnn, dcrnn_pred, train_time=dcrnn_train_time, model=dcrnn_model)

# A3T-GCN

### Additional Temporal Architecture — A3T-GCN

Unlike T-GCN/DCRNN's step-by-step recurrence, A3T-GCN's PyG-Temporal implementation expects a full **windowed "periods" tensor** per node (shape `[n_nodes, in_channels, periods]`) and applies attention across the period dimension internally — so it reuses the batched windowed data (`X_train_g`/`X_test_g`) like the static models, not the step-by-step loop.

In [ ]:
from torch_geometric_temporal.nn.recurrent import A3TGCN

class A3TGCNForecaster(nn.Module):
    def __init__(self, in_channels=1, hidden_channels=32, periods=WINDOW):
        super().__init__()
        self.a3tgcn = A3TGCN(in_channels, hidden_channels, periods)
        self.readout = nn.Linear(hidden_channels * n_nodes, 1)

    def forward(self, x, edge_index, edge_weight):
        h = self.a3tgcn(x, edge_index, edge_weight)
        return self.readout(h.flatten())

a3tgcn_model = A3TGCNForecaster().to(device)
optimizer_a3t = torch.optim.Adam(a3tgcn_model.parameters(), lr=1e-3)


train_start = time.time()
a3tgcn_model.train()
for epoch in range(5):
    epoch_loss = 0.0
    perm = torch.randperm(len(X_train_t))
    n_small_batches = min(n_batches, 50)
    for i in range(n_small_batches):
        idx = perm[i*batch_size:(i+1)*batch_size]
        xb, yb = X_train_t[idx].to(device), y_train_t[idx].to(device)
        optimizer_a3t.zero_grad()
        batch_preds = []
        for b in range(xb.shape[0]):
            x_in = xb[b].unsqueeze(1)
            pred = a3tgcn_model(x_in, edge_index_dev, edge_weight_dev)
            batch_preds.append(pred)
        batch_preds = torch.stack(batch_preds)
        loss = loss_fn(batch_preds, yb)
        loss.backward()
        optimizer_a3t.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={epoch_loss/n_small_batches:.6f}")
a3tgcn_train_time = time.time() - train_start



In [ ]:
a3tgcn_model.eval()
a3t_preds = []
with torch.no_grad():
    for i in range(0, len(X_test_t), batch_size):
        xb = X_test_t[i:i+batch_size].to(device)
        batch_preds = []
        for b in range(xb.shape[0]):
            x_in = xb[b].unsqueeze(1)
            batch_preds.append(a3tgcn_model(x_in, edge_index_dev, edge_weight_dev))
        a3t_preds.append(torch.stack(batch_preds).cpu().numpy())

a3t_pred_scaled = np.concatenate(a3t_preds).flatten()
a3t_pred = target_scaler_g.inverse_transform(a3t_pred_scaled.reshape(-1, 1)).flatten()

rmse, mae, r2 = eval_metrics(y_true_g, a3t_pred)
print(f"A3T-GCN — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}")
results_df = log_result(results_df, "A3T-GCN", y_true_g, a3t_pred, train_time=a3tgcn_train_time, model=a3tgcn_model)

# STGCN

### Additional Temporal Architecture — STGCN

Uses convolutional temporal blocks (`STConv`) rather than recurrence — expects a batched `[batch, seq_len, num_nodes, in_channels]` tensor. The output linear readout is built lazily on the first forward pass since STConv's temporal-convolution kernel shrinks the sequence length in a way that depends on kernel size/layer count.

In [ ]:
from torch_geometric_temporal.nn.attention import STConv

class STGCNForecaster(nn.Module):
    def __init__(self, num_nodes, in_channels=1, hidden_channels=16, out_channels=8, kernel_size=3, K=2):
        super().__init__()
        self.stconv = STConv(num_nodes, in_channels, hidden_channels, out_channels, kernel_size, K)
        self.readout = None

    def forward(self, x, edge_index, edge_weight):
        out = self.stconv(x, edge_index, edge_weight)
        out_flat = out.reshape(out.shape[0], -1)
        if self.readout is None:
            self.readout = nn.Linear(out_flat.shape[1], 1).to(out_flat.device)
        return self.readout(out_flat)

stgcn_model = STGCNForecaster(num_nodes=n_nodes).to(device)

X_train_stgcn = torch.tensor(X_train_g, dtype=torch.float32).permute(0, 2, 1).unsqueeze(-1)
X_test_stgcn = torch.tensor(X_test_g, dtype=torch.float32).permute(0, 2, 1).unsqueeze(-1)

optimizer_st = torch.optim.Adam(stgcn_model.parameters(), lr=1e-3)

train_start=time.time() 
stgcn_model.train()
for epoch in range(5):
    epoch_loss = 0.0
    perm = torch.randperm(len(X_train_stgcn))
    for i in range(n_batches):
        idx = perm[i*batch_size:(i+1)*batch_size]
        xb = X_train_stgcn[idx].to(device)
        yb = y_train_t[idx].to(device)
        optimizer_st.zero_grad()
        pred = stgcn_model(xb, edge_index_dev, edge_weight_dev)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer_st.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={epoch_loss/n_batches:.6f}")
stgcn_train_time = time.time() - train_start



In [ ]:
stgcn_model.eval()
preds = []
with torch.no_grad():
    for i in range(0, len(X_test_stgcn), batch_size):
        xb = X_test_stgcn[i:i+batch_size].to(device)
        preds.append(stgcn_model(xb, edge_index_dev, edge_weight_dev).cpu().numpy())
stgcn_pred_scaled = np.concatenate(preds).flatten()
stgcn_pred = target_scaler_g.inverse_transform(stgcn_pred_scaled.reshape(-1, 1)).flatten()

rmse, mae, r2 = eval_metrics(y_true_g, stgcn_pred)
print(f"STGCN — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}")
results_df = log_result(results_df, "STGCN", y_true_g, stgcn_pred, train_time=stgcn_train_time, model=stgcn_model)

# LSTM-GNN Hybrid

Custom combination (no single ready-made module covers this): a GCN produces a spatial embedding of the 6-node graph at each individual timestep within the window, and the resulting sequence of per-timestep graph embeddings is fed into an LSTM to model temporal dependence. This is the standard "GNN handles space, LSTM handles time" hybrid pattern. Note: this loops over both batch and window-timestep, so it's the slowest model in this notebook — epochs and training data are subsampled deliberately.

In [ ]:
class LSTMGNNHybrid(nn.Module):
    def __init__(self, in_channels=1, gnn_hidden=16, lstm_hidden=32):
        super().__init__()
        self.gcn = GCNConv(in_channels, gnn_hidden)
        self.lstm = nn.LSTM(gnn_hidden * n_nodes, lstm_hidden, batch_first=True)
        self.readout = nn.Linear(lstm_hidden, 1)

    def forward(self, x, edge_index, edge_weight):
        batch_size, _, window = x.shape
        seq_embeddings = []
        for t in range(window):
            xt = x[:, :, t].unsqueeze(-1)
            embeds = []
            for b in range(batch_size):
                h = F.relu(self.gcn(xt[b], edge_index, edge_weight))
                embeds.append(h.flatten())
            seq_embeddings.append(torch.stack(embeds))
        seq = torch.stack(seq_embeddings, dim=1)
        _, (h_n, _) = self.lstm(seq)
        return self.readout(h_n[-1])

lstm_gnn_model = LSTMGNNHybrid().to(device)
optimizer_lg = torch.optim.Adam(lstm_gnn_model.parameters(), lr=1e-3)

# Subsample training data — this model's nested loop makes full-dataset training
# impractical within a normal notebook session; state this explicitly in your report.
subsample_idx = np.random.RandomState(42).choice(len(X_train_t), size=min(3000, len(X_train_t)), replace=False)
X_train_sub = X_train_t[subsample_idx]
y_train_sub = y_train_t[subsample_idx]


train_start=time.time() 
lstm_gnn_model.train()
sub_batch_size = 64
n_sub_batches = len(X_train_sub) // sub_batch_size
for epoch in range(5):
    epoch_loss = 0.0
    perm = torch.randperm(len(X_train_sub))
    for i in range(n_sub_batches):
        idx = perm[i*sub_batch_size:(i+1)*sub_batch_size]
        xb, yb = X_train_sub[idx].to(device), y_train_sub[idx].to(device)
        optimizer_lg.zero_grad()
        pred = lstm_gnn_model(xb, edge_index_dev, edge_weight_dev)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer_lg.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={epoch_loss/n_sub_batches:.6f}")

lstmgnn_train_time = time.time() - train_start

In [ ]:
lstm_gnn_model.eval()
test_subsample_idx = np.random.RandomState(43).choice(len(X_test_t), size=min(1000, len(X_test_t)), replace=False)
X_test_sub = X_test_t[test_subsample_idx]
y_test_sub_true = target_scaler_g.inverse_transform(y_test_g[test_subsample_idx].reshape(-1, 1)).flatten()

preds = []
with torch.no_grad():
    for i in range(0, len(X_test_sub), sub_batch_size):
        xb = X_test_sub[i:i+sub_batch_size].to(device)
        preds.append(lstm_gnn_model(xb, edge_index_dev, edge_weight_dev).cpu().numpy())
lstmgnn_pred_scaled = np.concatenate(preds).flatten()
lstmgnn_pred = target_scaler_g.inverse_transform(lstmgnn_pred_scaled.reshape(-1, 1)).flatten()

rmse, mae, r2 = eval_metrics(y_test_sub_true, lstmgnn_pred)
print(f"LSTM-GNN Hybrid [evaluated on {len(test_subsample_idx)}-point subsample] — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}")
results_df = log_result(results_df, "LSTM-GNN Hybrid", y_test_sub_true, lstmgnn_pred,fold="holdout_subsample", train_time=lstmgnn_train_time, model=lstm_gnn_model)

# DySAT-Inspired Model 

**Important caveat, state this plainly in your report**: DySAT (Dynamic Self-Attention Network) is not available as a ready module in `torch_geometric` or `torch_geometric_temporal`, and a full reimplementation of the published architecture (structural attention + temporal self-attention across dynamic graph snapshots) is a substantial undertaking beyond this project's scope. What's implemented below is a **simplified, DySAT-inspired approximation**: temporal self-attention (via a small Transformer encoder) applied to each node's own window sequence, followed by GAT-style structural attention across nodes — capturing the *spirit* of "attention over both space and time" without claiming to reproduce DySAT's actual mechanism. This is a legitimate learning exercise but should be labeled as an approximation, not the real DySAT, if referenced in your write-up.

In [ ]:
class DySATInspired(nn.Module):
    def __init__(self, window, hidden_channels=16, n_heads=2):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(d_model=1, nhead=1, dim_feedforward=32, batch_first=True)
        self.temporal_attn = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.temporal_proj = nn.Linear(window, hidden_channels)
        self.gat = GATConv(hidden_channels, hidden_channels, heads=n_heads, edge_dim=1)
        self.readout = nn.Linear(hidden_channels * n_heads * n_nodes, 1)

    def forward(self, x, edge_index, edge_weight):
        batch_size = x.shape[0]
        edge_attr = edge_weight.unsqueeze(-1)
        outs = []
        for b in range(batch_size):
            xb = x[b].unsqueeze(-1)
            temporal_out = self.temporal_attn(xb).squeeze(-1)
            node_feats = F.relu(self.temporal_proj(temporal_out))
            h = F.elu(self.gat(node_feats, edge_index, edge_attr))
            outs.append(h.flatten())
        return self.readout(torch.stack(outs))

dysat_model = DySATInspired(window=WINDOW).to(device)
optimizer_dy = torch.optim.Adam(dysat_model.parameters(), lr=1e-3)

train_start = time.time() 
dysat_model.train()
for epoch in range(5):
    epoch_loss = 0.0
    perm = torch.randperm(len(X_train_sub))
    for i in range(n_sub_batches):
        idx = perm[i*sub_batch_size:(i+1)*sub_batch_size]
        xb, yb = X_train_sub[idx].to(device), y_train_sub[idx].to(device)
        optimizer_dy.zero_grad()
        pred = dysat_model(xb, edge_index_dev, edge_weight_dev)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer_dy.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={epoch_loss/n_sub_batches:.6f}")
dysat_train_time = time.time() - train_start


In [ ]:
dysat_model.eval()
preds = []
with torch.no_grad():
    for i in range(0, len(X_test_sub), sub_batch_size):
        xb = X_test_sub[i:i+sub_batch_size].to(device)
        preds.append(dysat_model(xb, edge_index_dev, edge_weight_dev).cpu().numpy())
dysat_pred_scaled = np.concatenate(preds).flatten()
dysat_pred = target_scaler_g.inverse_transform(dysat_pred_scaled.reshape(-1, 1)).flatten()

rmse, mae, r2 = eval_metrics(y_test_sub_true, dysat_pred)
print(f"DySAT-inspired [evaluated on {len(test_subsample_idx)}-point subsample] — RMSE: {rmse:.4f}  MAE: {mae:.4f}  R2: {r2:.4f}")
results_df = log_result(results_df, "DySAT-inspired", y_test_sub_true, dysat_pred,fold="holdout_subsample", train_time=dysat_train_time, model=dysat_model)

# All Architectures Summary

Reminder when interpreting this table: **GAT, GCN, GraphSAGE, GIN, T-GCN, DCRNN, A3T-GCN, STGCN** are all evaluated on the full test set; **LSTM-GNN Hybrid** and **DySAT-inspired** are evaluated on a random subsample (flagged via the `fold` column) due to their nested per-sample computation cost — do not directly compare their RMSE/MAE to the full-test-set models without noting this in your report.

In [ ]:
print(results_df[results_df.model.isin([
    "GAT", "GCN", "GraphSAGE", "GIN", "T-GCN", "DCRNN", "A3T-GCN", "STGCN",
    "LSTM-GNN Hybrid", "DySAT-inspired"
])].to_string(index=False))

### Robustness Test — Synthetic Sensor-Failure & Heatwave Corruption

The instructor note requires robustness to heatwaves and sensor failures. The real dataset only contains **one** genuine flatline event and its "heatwave spikes" turned out to be a resampling artifact (Phase 1.1) — not enough naturally-occurring failure cases to evaluate robustness honestly. Instead, we build a **corrupted copy of the test set** with controlled, synthetic injections (flatlines, heatwave-style spikes, sensor noise bursts) and compare each model's error on clean vs. corrupted data. This is a real, controlled experiment rather than relying on the one naturally-occurring event.

In [ ]:
def inject_corruption(df_in, seed=42, n_flatlines=5, n_spikes=5, n_noise_bursts=5,
                       flatline_len=10, spike_magnitude=15, noise_std=5):
    rng = np.random.RandomState(seed)
    corrupted = df_in.copy().reset_index(drop=True)
    n = len(corrupted)
    log = []

    # Flatlines: freeze Soil_Moisture for a window
    for _ in range(n_flatlines):
        start = rng.randint(0, n - flatline_len)
        frozen_val = corrupted.loc[start, "Soil_Moisture"]
        corrupted.loc[start:start+flatline_len-1, "Soil_Moisture"] = frozen_val
        log.append(("flatline", start, start+flatline_len-1))

    # Heatwave-style spikes: sharp temperature jump held for a short window
    for _ in range(n_spikes):
        start = rng.randint(0, n - flatline_len)
        corrupted.loc[start:start+flatline_len-1, "Temperature"] += spike_magnitude
        log.append(("spike", start, start+flatline_len-1))

    # Sensor noise bursts: inject gaussian noise into Soil_Moisture for a window
    for _ in range(n_noise_bursts):
        start = rng.randint(0, n - flatline_len)
        noise = rng.normal(0, noise_std, flatline_len)
        corrupted.loc[start:start+flatline_len-1, "Soil_Moisture"] += noise
        log.append(("noise", start, start+flatline_len-1))

    return corrupted, log

test_df_corrupted, corruption_log = inject_corruption(test_df)
print(f"Injected {len(corruption_log)} corruption events:")
for kind, s, e in corruption_log:
    print(f"  {kind:10s} rows {s}-{e}")

In [ ]:
# Evaluate GAT on corrupted vs clean test data (reuses the already-trained gat_model)
test_scaled_corrupt = {}
for col in graph_nodes:
    test_scaled_corrupt[col] = node_scalers[col].transform(test_df_corrupted[[col]]).flatten()
test_target_scaled_corrupt = target_scaler_g.transform(test_df_corrupted[[target_col]]).flatten()

X_test_corrupt, y_test_corrupt = build_windows(test_scaled_corrupt, test_target_scaled_corrupt, WINDOW)
X_test_corrupt_t = torch.tensor(X_test_corrupt, dtype=torch.float32)

gat_model.eval()
preds_corrupt = []
with torch.no_grad():
    for i in range(0, len(X_test_corrupt_t), batch_size):
        xb = X_test_corrupt_t[i:i+batch_size].to(device)
        preds_corrupt.append(gat_model(xb, edge_index_dev, edge_weight_dev).cpu().numpy())
gat_pred_corrupt_scaled = np.concatenate(preds_corrupt).flatten()
gat_pred_corrupt = target_scaler_g.inverse_transform(gat_pred_corrupt_scaled.reshape(-1, 1)).flatten()
y_true_corrupt = target_scaler_g.inverse_transform(y_test_corrupt.reshape(-1, 1)).flatten()

rmse_clean, mae_clean, r2_clean = eval_metrics(y_true_g, gat_pred)
rmse_corrupt, mae_corrupt, r2_corrupt = eval_metrics(y_true_corrupt, gat_pred_corrupt)

print("GAT robustness comparison:")
print(f"  Clean:     RMSE={rmse_clean:.4f}  MAE={mae_clean:.4f}  R2={r2_clean:.4f}")
print(f"  Corrupted: RMSE={rmse_corrupt:.4f}  MAE={mae_corrupt:.4f}  R2={r2_corrupt:.4f}")
print(f"  RMSE degradation: {(rmse_corrupt - rmse_clean) / rmse_clean * 100:.1f}%")

### Irrigation Decision Trigger — Threshold Rule

`irrigation_status` has only 10 positive rows in the entire dataset — not enough to train a classifier. Per the instructor note's requirement to "support resilient smart irrigation decisions," we apply a **data-driven threshold rule** on top of the model's soil-moisture forecast instead: if predicted moisture falls below a low percentile of the observed distribution, flag irrigation as recommended.

In [ ]:
# Data-driven threshold: 10th and 25th percentile of observed Soil_Moisture (train set)
threshold_10 = train_df["Soil_Moisture"].quantile(0.10)
threshold_25 = train_df["Soil_Moisture"].quantile(0.25)
print(f"10th percentile threshold: {threshold_10:.2f}%")
print(f"25th percentile threshold: {threshold_25:.2f}%")

# Apply to GAT's clean-test predictions as the example decision layer
irrigation_flag_10 = (gat_pred < threshold_10).astype(int)
irrigation_flag_25 = (gat_pred < threshold_25).astype(int)

print(f"\nIrrigation recommended (10th pct threshold): {irrigation_flag_10.sum()} of {len(irrigation_flag_10)} intervals ({irrigation_flag_10.mean()*100:.2f}%)")
print(f"Irrigation recommended (25th pct threshold): {irrigation_flag_25.sum()} of {len(irrigation_flag_25)} intervals ({irrigation_flag_25.mean()*100:.2f}%)")

plt.figure(figsize=(14, 4))
plt.plot(y_true_g[:2000], label="Actual soil moisture", alpha=0.7)
plt.axhline(threshold_10, color="red", linestyle="--", label=f"10th pct threshold ({threshold_10:.1f}%)")
plt.axhline(threshold_25, color="orange", linestyle="--", label=f"25th pct threshold ({threshold_25:.1f}%)")
plt.title("Irrigation Trigger Threshold vs Actual Soil Moisture (first 2000 test points)")
plt.legend()
plt.tight_layout()
plt.show()

## Final Results Summary

In [ ]:
print(results_df.to_string(index=False))